In [1]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [2]:
import numpy as np
from PIL import Image

datadir = "dataset"
imgsize = 120
validext = (".jpg", ".jpeg", ".png")

def loadimagesfromfolder(folder, label):
    images, labels = [], []
    folderpath = os.path.join(datadir, folder)

    for fname in os.listdir(folderpath):
        if not fname.lower().endswith(validext):
            continue
        imgpath = os.path.join(folderpath, fname)
        try:
            img = Image.open(imgpath).convert("RGB").resize((imgsize, imgsize))
            images.append(np.array(img))
            labels.append(label)
        except Exception as e:
            print(f"skipping {fname}: {e}")

    return images, labels


freshimages, freshlabels = loadimagesfromfolder("fresh", 1)
rottenimages, rottenlabels = loadimagesfromfolder("rotten", 0)

x = np.array(freshimages + rottenimages, dtype="float32") / 255.0
y = np.array(freshlabels + rottenlabels, dtype="int32")

print(f"loaded {len(x)} images -> shape {x.shape}, labels {y.shape}")

loaded 81 images -> shape (81, 120, 120, 3), labels (81,)


In [3]:
from sklearn.model_selection import train_test_split

xtrain, xtemp, ytrain, ytemp = train_test_split(
    x, y, test_size=0.30, random_state=42, stratify=y
)
xval, xtest, yval, ytest = train_test_split(
    xtemp, ytemp, test_size=0.50, random_state=42, stratify=ytemp
)

print(f"train {xtrain.shape}, val {xval.shape}, test {xtest.shape}")

train (56, 120, 120, 3), val (12, 120, 120, 3), test (13, 120, 120, 3)


In [4]:
dataaugmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [5]:
model = models.Sequential([
    layers.Input(shape=(imgsize, imgsize, 3)),
    dataaugmentation,

    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dropout(0.4),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 120, 120, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 120, 120, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 60, 60, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 60, 60, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 30, 30, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 7200)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 7200)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       460,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 475,265 (1.81 MB)

 Trainable params: 475,265 (1.81 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
epochs = 25

earlystop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

history = model.fit(
    xtrain, ytrain,
    validation_data=(xval, yval),
    epochs=epochs,
    batch_size=16,
    callbacks=[earlystop]
)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 7s 315ms/step - accuracy: 0.5000 - loss: 0.8279 - val_accuracy: 0.4167 - val_loss: 0.7124
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 218ms/step - accuracy: 0.4464 - loss: 0.7019 - val_accuracy: 0.5000 - val_loss: 0.6673
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 221ms/step - accuracy: 0.6429 - loss: 0.6571 - val_accuracy: 0.5833 - val_loss: 0.6016
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 148ms/step - accuracy: 0.5536 - loss: 0.6537 - val_accuracy: 0.8333 - val_loss: 0.4875
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 148ms/step - accuracy: 0.6607 - loss: 0.6037 - val_accuracy: 0.9167 - val_loss: 0.4048
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 143ms/step - accuracy: 0.6964 - loss: 0.5710 - val_accuracy: 0.8333 - val_loss: 0.3626
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 149ms/step - accuracy: 0.7679 - loss: 0.4877 - val_accuracy: 0.9167 - val_loss: 0.3041
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 145ms/step - accuracy: 0.6786 - loss: 0.5180 - val_accuracy: 0.9167 - val_loss:

In [7]:
testloss, testacc = model.evaluate(xtest, ytest)
print(f"test accuracy: {testacc:.3f}, test loss: {testloss:.3f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 386ms/step - accuracy: 0.9231 - loss: 0.3880
test accuracy: 0.923, test loss: 0.388


In [8]:
from sklearn.metrics import classification_report, confusion_matrix

classnames = ["Rotten", "Fresh"]

ypredprob = model.predict(xtest)
ypred = (ypredprob > 0.5).astype(int).flatten()

print(classification_report(ytest, ypred, target_names=classnames))
print("confusion matrix")
print(confusion_matrix(ytest, ypred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 362ms/step
              precision    recall  f1-score   support

      Rotten       1.00      0.86      0.92         7
       Fresh       0.86      1.00      0.92         6

    accuracy                           0.92        13
   macro avg       0.93      0.93      0.92        13
weighted avg       0.93      0.92      0.92        13

confusion matrix
[[6 1]
 [0 6]]


In [9]:
model.save("orange_model.keras")